# PubMed Biomedical Metadata — Exploratory Analysis

A tour of the dataset: scale, coverage over time, journals, MeSH topics, and how completeness of fields (keywords, conflict-of-interest statements) changes across the decades.

## Setup

## MOVE HERE IF IN NEED TO RUN CODE

- [2b. Trim the live edge](#2b-trim-the-live-edge-drop-2026)

In [ ]:
import os, glob, collections
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING — pick ONE option
# =====================================================================

# ---- OPTION A: LOCAL (active) -----------------------------------------
# Notebook runs from notebooks/, data sits one level up in the project root.
# Loading data with 2026 records for checking porposes
ROOT = os.path.dirname(os.getcwd())                       # go up one folder
DATA_DIR = os.path.join(ROOT, "data", "2_clean")          # full clean corpus (with abstracts)

# ---- OPTION B: KAGGLE (commented out — uncomment when running on Kaggle) ----
# # Data on Kaggle is filtered and has data up to 2025 already, so we can load the whole dataset for analysis.
# # The attached dataset lives under /kaggle/input/<dataset-slug>/.
# # This auto-detects the folder of Parquet shards so no path editing is needed.
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input — attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])

# =====================================================================
df = pd.read_parquet(DATA_DIR)
print(f"loaded {len(df):,} records from {DATA_DIR}")
print("columns:", list(df.columns))
df.head(3)

## 1. Scale and schema
The dataset is one row per article, keyed by PubMed ID (`uid`). Each record carries bibliographic metadata and flattened list fields (authors, affiliations, MeSH descriptors, keywords).

In [ ]:
print(f"records:       {len(df):,}")
print(f"unique PMIDs:  {df['uid'].nunique():,}")
print(f"year range:    {int(df['year'].min())}–{int(df['year'].max())}")
print(f"columns:       {df.shape[1]}")
df.dtypes

## 2. Publications per year
Annual volume grows steadily from the mid-1990s, accelerating through the 2010s, with a conspicuous dip around 2012–2015 (highlighted) and a small live-edge tail at the most recent year.

In [ ]:
per_year = df['year'].value_counts().sort_index()

plt.figure(figsize=(12, 4))
sns.lineplot(x=per_year.index, y=per_year.values, marker="o", color="#1d6fb8")

# highlight the 2012–2015 anomaly window
plt.axvspan(2012, 2015, color="#e07a5f", alpha=0.15)          # shaded band
plt.axvline(2012, color="#e07a5f", ls="--", lw=1.2)            # left marker
plt.axvline(2015, color="#e07a5f", ls="--", lw=1.2)            # right marker

# overlay the dip segment in a different color
seg = per_year.loc[2012:2015]
sns.lineplot(x=seg.index, y=seg.values, marker="o", color="#e07a5f", lw=2.5)

plt.title("Articles per year (2012–2015 dip highlighted)")
plt.xlabel("year"); plt.ylabel("articles")
plt.tight_layout(); plt.show()

### The 2012–2015 dip

Article counts fall sharply from a 2012 peak to a 2014 low, then recover. This is very likely an **artifact of the affiliation filter**, not a real decline in output: PubMed's recording of author affiliations changed around 2013–2014 (all-author affiliations were captured more completely from ~2014 onward), and a query that filters on US affiliation under-matches records in the transition years. Actual biomedical publishing did not drop ~40% and rebound; the dip reflects how affiliation metadata was indexed, so counts in this window should be treated with caution.

## 2b. Trim the live edge (drop 2026)

The publications-per-year chart above shows a tiny tail in 2026: a handful of records carrying electronic / ahead-of-print dates, plus the still-accruing final months. These are the "live edge" of PubMed — not stable or reproducible, and a misleading sliver on any time series. For a clean, frozen 1994–2025 view, they are dropped here. This affects only this analysis; the underlying clean corpus is unchanged.

Already in export folder 

In [ ]:
import os, glob, collections
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = os.path.dirname(os.getcwd())                       # go up one folder
DATA_DIR = os.path.join(ROOT, "data", "2_clean")          # full clean corpus (with abstracts)

df = pd.read_parquet(os.path.join(ROOT, "data", "2_export", "abstracts_none"))

In [ ]:
per_year = df['year'].value_counts().sort_index()

plt.figure(figsize=(12, 4))
sns.lineplot(x=per_year.index, y=per_year.values, marker="o", color="#1d6fb8")

# highlight the 2012–2015 anomaly window
plt.axvspan(2012, 2015, color="#e07a5f", alpha=0.15)          # shaded band
plt.axvline(2012, color="#e07a5f", ls="--", lw=1.2)            # left marker
plt.axvline(2015, color="#e07a5f", ls="--", lw=1.2)            # right marker

# overlay the dip segment in a different color
seg = per_year.loc[2012:2015]
sns.lineplot(x=seg.index, y=seg.values, marker="o", color="#e07a5f", lw=2.5)

plt.title("Articles per year (2012–2015 dip highlighted)")
plt.xlabel("year"); plt.ylabel("articles")
plt.tight_layout(); plt.show()

## 2c. Diagnosing the 2012–2015 dip

Article counts fall sharply from a 2012 peak to a 2014 low (−32.5%), then recover — against otherwise steady 2–6%/year growth. The cells below establish the magnitude and isolate the cause. Note that an affiliation-coverage check is **circular** here (the query requires a US affiliation, so coverage is ~100% by construction) and cannot detect the dip; it is shown only to make that explicit.

In [ ]:
per_year = df['year'].value_counts().sort_index()
stats = pd.DataFrame({"articles": per_year})
stats["yoy_change"] = stats["articles"].diff().astype("Int64")
stats["yoy_pct"] = (stats["articles"].pct_change() * 100).round(1)
print(stats.to_string())

In [ ]:
cov = df.assign(has_aff=df['affiliations'].map(lambda v: len(v) > 0)).groupby('year')['has_aff'].mean() * 100
print(cov.round(3).to_string())
print("\nNote: ~100% every year is expected — the query requires an affiliation, so this cannot detect the dip.")

In [ ]:
# decompose the dip by date-precision: does it hit all categories together?
prec_counts = df.groupby(['year', 'pubdate_precision']).size().unstack(fill_value=0)
plt.figure(figsize=(12, 4))
for col in prec_counts.columns:
    sns.lineplot(x=prec_counts.index, y=prec_counts[col], marker="o", label=col)
plt.axvspan(2012, 2015, color="#e07a5f", alpha=0.10)
plt.title("Article counts by date-precision, per year")
plt.xlabel("year"); plt.ylabel("articles"); plt.legend(title="precision")
plt.tight_layout(); plt.show()
print(prec_counts.loc[2011:2016].to_string())

In [ ]:
# decompose by journal: is the dip uniform (systemic) or concentrated in a few sources?
top_j = df['journal'].value_counts().head(15).index
piv = df[df['journal'].isin(top_j)].groupby(['year', 'journal']).size().unstack(fill_value=0)
ratio = (piv.loc[2014] / piv.loc[2012]).sort_values()
print("2014/2012 article ratio for top-15 journals:")
print(ratio.round(2).to_string())
print(f"\nmedian ratio: {ratio.median():.2f}  "
      f"(uniform ~same = systemic; wide spread = specific journals drove it)")

In [ ]:
# affiliation coverage — shown but CIRCULAR (query requires affiliation), cannot detect the dip
cov = df.assign(has_aff=df['affiliations'].map(lambda v: len(v) > 0)).groupby('year')['has_aff'].mean() * 100
print(cov.round(3).to_string())
print("\nNote: ~100% every year is expected — the query requires an affiliation, so this cannot detect the dip.")

**Conclusion — the dip is an affiliation-filter artifact, not a real decline.** Decomposing by date precision shows all three precision types drop ~35% together in 2014, so it is not a date-format issue. Decomposing by journal reveals the cause: established US journals nearly vanish from the 2014 bucket (Journal of Biological Chemistry, Blood, Cancer ≈ 0.00× their 2012 count; PNAS 0.01×) while newly launched megajournals grow (Scientific Reports 4×, Nature Communications 6.5×). Those established journals plainly did not stop publishing US research in 2014 — the `USA[Affiliation]` filter simply failed to match their 2013–2014 records. This coincides with PubMed's affiliation-indexing transition (~2013–2014, when MEDLINE moved from first-author-only to all-author affiliations and restructured the affiliation field). The harvest matches PubMed's own per-month counts exactly, so the dip is a faithful property of the filtered query — a metadata artifact, **not** a real change in output. Volume in 2013–2015 should not be read as a trend, and rate/volume analyses should treat the window as unreliable.

## 3. Date precision
Most records carry imprecise publication dates (year-only or year+month). The `pubdate_precision` flag records this; month-level time analysis should use only `full_date` / `year_month` records.

In [ ]:
prec = df['pubdate_precision'].value_counts()
plt.figure(figsize=(7, 4))
sns.barplot(x=prec.index, y=prec.values, color="#1d6fb8")
plt.title("Publication-date precision"); plt.ylabel("records"); plt.xlabel("")
plt.tight_layout(); plt.show()
print((prec / len(df) * 100).round(1).astype(str) + " %")

## 4. Top journals
The corpus spans thousands of journals with a long tail — it is broad rather than dominated by a few high-volume titles.

In [ ]:
print("distinct journals:", df['journal'].nunique())
top = df['journal'].value_counts().head(15)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, color="#1d6fb8")
plt.title("Top 15 journals by article count"); plt.xlabel("articles"); plt.ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
total = len(df)
top_counts = df['journal'].value_counts()
print("journal concentration:")
for n in [5, 10, 20, 50, 100]:
    print(f"  top {n:>3} journals = {top_counts.head(n).sum()/total*100:5.1f}% of all articles")

# first appearance in the corpus (proxy for new journals; not true founding year)
first_seen = df.groupby('journal')['year'].min()
new_per_year = first_seen.value_counts().sort_index()
plt.figure(figsize=(12, 4))
sns.lineplot(x=new_per_year.index, y=new_per_year.values, marker="o", color="#2a9d5c")
plt.title("Journals by first appearance in the corpus (proxy, not founding year)")
plt.xlabel("year"); plt.ylabel("journals first seen"); plt.tight_layout(); plt.show()
print(new_per_year.tail(10).to_string())

## 5. Authors per paper
Author counts are right-skewed: most papers have a handful of authors, with a long tail of large collaborations. The composition shifts markedly over time toward larger teams.

In [ ]:
print(df['n_authors'].describe().astype(int))
plt.figure(figsize=(10, 4))
sns.histplot(df['n_authors'].clip(upper=30), bins=30, color="#1d6fb8")
plt.title("Authors per paper (clipped at 30)"); plt.xlabel("authors"); plt.ylabel("papers")
plt.tight_layout(); plt.show()

In [ ]:
band = pd.cut(df['n_authors'], [0, 1, 5, 20, 10**9], labels=["solo", "2–5", "6–20", "21+"])
collab = df.assign(band=band).groupby(['year', 'band'], observed=True).size().unstack(fill_value=0)
collab_pct = collab.div(collab.sum(axis=1), axis=0) * 100

collab_pct.plot.area(figsize=(12, 4), color=["#d9534f", "#1d6fb8", "#2a9d5c", "#8a5fb0"])
plt.title("Team-size composition by year (%)"); plt.xlabel("year"); plt.ylabel("% of articles")
plt.legend(title="authors", loc="lower left"); plt.tight_layout(); plt.show()
print("share by band, selected years:")
print(collab_pct.loc[collab_pct.index.isin([1995, 2005, 2015, 2025])].round(1).to_string())

## 6. MeSH topics
MeSH (Medical Subject Headings) are NLM's controlled vocabulary. The most frequent descriptors reflect the human-subject scope; the number of descriptors per article changed structurally over time.

In [ ]:
mc = collections.Counter(d for lst in df['mesh_descriptors'] for d in lst)
print("distinct MeSH descriptors:", len(mc))
top = pd.Series(dict(mc.most_common(15)))[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, color="#2a9d5c")
plt.title("Top 15 MeSH descriptors"); plt.xlabel("occurrences"); plt.ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
# top 15 descriptors over time
TOP_N = 15
top_descriptors = [d for d, _ in mc.most_common(TOP_N)]
mesh_exploded = (df[["year", "mesh_descriptors"]]
                 .explode("mesh_descriptors")
                 .rename(columns={"mesh_descriptors": "descriptor"}))
mesh_exploded = mesh_exploded[mesh_exploded["descriptor"].isin(top_descriptors)]
mesh_ts = mesh_exploded.groupby(["year", "descriptor"]).size().reset_index(name="count")

fig, ax = plt.subplots(figsize=(13, 6))
for desc in top_descriptors:
    sub = mesh_ts[mesh_ts["descriptor"] == desc]
    ax.plot(sub["year"], sub["count"], label=desc, linewidth=1.5)
ax.set_title("Top 15 MeSH descriptors — article count per year")
ax.set_xlabel("year"); ax.set_ylabel("articles")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# mean descriptors per article over time — the structural shift
mesh_year = df.groupby('year')['n_mesh'].mean()
plt.figure(figsize=(12, 4))
sns.lineplot(x=mesh_year.index, y=mesh_year.values, marker="o", color="#2a9d5c")
plt.title("Mean MeSH descriptors per article, by year"); plt.xlabel("year"); plt.ylabel("mean # MeSH")
plt.tight_layout(); plt.show()
print(mesh_year.round(2).to_string())

In [ ]:
# how many articles per year have zero MeSH vs at least one
mesh_coverage = df.groupby("year").agg(
    total=("uid", "count"),
    with_mesh=("n_mesh", lambda x: (x > 0).sum()),
    avg_mesh=("n_mesh", "mean")
).reset_index()
mesh_coverage["pct_with_mesh"] = mesh_coverage["with_mesh"] / mesh_coverage["total"] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(mesh_coverage["year"], mesh_coverage["total"], label="total articles")
axes[0].set_title("total articles per year"); axes[0].set_xlabel("year")
axes[1].plot(mesh_coverage["year"], mesh_coverage["avg_mesh"], color="orange")
axes[1].set_title("avg MeSH descriptors per article per year"); axes[1].set_xlabel("year")
plt.tight_layout(); plt.show()

print(mesh_coverage[mesh_coverage["year"].between(2011, 2025)])

MeSH coverage is 100% across all years — every record carries at least one descriptor (PubMed releases fully-indexed records). However, the **mean number of descriptors per article is stable at ~13 through 2019, then falls to ~8 by 2022–2023**, with a partial rebound in 2024–2025. This matches NLM's transition to automated indexing (~2021–2022, which assigns fewer descriptors), compounded by indexing lag on the most recent years — not a change in article content. Analyses relying on MeSH breadth (e.g. topic co-occurrence) should segment by era and treat post-2019 depth cautiously.

## 7. Field completeness over time
Several fields became standard in PubMed only gradually: keywords (from ~2012) and conflict-of-interest statements (from ~2017). Their coverage by year is a key caveat for any longitudinal analysis.

In [ ]:
by_year = df.assign(
    has_kw=df['n_keywords'] > 0,
    has_mesh=df['n_mesh'] > 0,
).groupby('year').agg(
    keywords=('has_kw', 'mean'),
    coi=('has_coi', 'mean'),
    mesh=('has_mesh', 'mean'),
) * 100

plt.figure(figsize=(12, 4))
sns.lineplot(data=by_year, dashes=False, markers=False)
plt.title("Field coverage by year (%)"); plt.xlabel("year"); plt.ylabel("% of records")
plt.legend(title=""); plt.tight_layout(); plt.show()

# crossover points that make the "valid from" thresholds concrete
kw_cross = by_year.index[by_year['keywords'] >= 50].min()
coi_first = by_year.index[by_year['coi'] >= 5].min()
print(f"keywords first reach 50% coverage: {kw_cross}")
print(f"COI statements first reach 5% coverage: {coi_first}")

## 8. Text fields (titles and abstracts)
The published release carries no abstract text, but titles are always present and abstract *length* is retained, so both can be characterised. (Run locally on the clean corpus, abstract text is present and these read the real distributions.)

In [ ]:
print("abstract text included:", (df['abstract'].fillna('').str.len() > 0).any())
print(f"mean original abstract length: {df['abstract_len'].mean():.0f} chars")
plt.figure(figsize=(10, 3))
sns.histplot(df['abstract_len'].clip(upper=4000), bins=50, color="#8a5fb0")
plt.title("Original abstract length (chars, clipped 4000)"); plt.xlabel("chars")
plt.tight_layout(); plt.show()

In [ ]:
tl = df['title'].fillna('').str.len()
print(f"title length — mean {tl.mean():.0f}, median {int(tl.median())}, "
      f"min {int(tl.min())}, max {int(tl.max())} chars")
plt.figure(figsize=(10, 3))
sns.histplot(tl.clip(upper=300), bins=60, color="#1d6fb8")
plt.title("Title length (chars)"); plt.xlabel("chars"); plt.tight_layout(); plt.show()

abs_year = df.groupby('year')['abstract_len'].mean()
plt.figure(figsize=(12, 4))
sns.lineplot(x=abs_year.index, y=abs_year.values, marker="o", color="#8a5fb0")
plt.title("Mean abstract length over time (chars)"); plt.xlabel("year"); plt.ylabel("mean chars")
plt.tight_layout(); plt.show()
print(abs_year.round(0).astype(int).to_string())

## 9. Recommendations for downstream analysis

This corpus is metadata-rich but has structural quirks that will bias naive analysis. Each note flags what to control for.

### Cross-cutting (apply to almost everything)
- **The 2013–2015 volume dip is an affiliation-filter artifact, not real** (established journals vanish from the 2014 US-affiliation bucket while still publishing; coincides with PubMed's ~2013–2014 affiliation-indexing change). Never read it as a publishing trend; for rate/volume work, exclude or flag 2013–2015.
- **~77% of records have imprecise dates** (year or year+month) — use `pubdate_precision`; restrict to `full_date` for month-level work.
- **No abstract text in the published release** — text methods (NER, topic modelling, sentiment) need the local abstract-bearing build or a re-fetch by PMID.
- **Author names are not disambiguated** — "J Smith" is not unified across records.

### 05 — Keywords
- Near-zero before 2012, ~80% coverage only after 2017; longitudinal keyword analysis valid from ~2013 onward.
- Early keyword records contain non-biomedical noise (e.g. NASA metadata strings) — filter before use.

### 06 — MeSH
- Descriptors per article are stable ~13 through 2019, then fall to ~8 by 2022–2023 (NLM automated indexing, not a content change). Co-occurrence and breadth measures differ pre/post; segment by era and treat 2020+ depth cautiously.
- "Humans", "Female", "Male", "Adult" dominate and are uninformative for clustering — remove before similarity/network analysis.

### 07 — COVID-19 time series
- "COVID-19" MeSH exists only from 2020; union with "Coronavirus Infections", "SARS Virus", "Betacoronavirus" for the pre-2020 baseline.
- The 2020–2021 spike is real but inflated by rapid indexing; report absolute counts alongside shares.

### 08 — Co-authorship network
- Big-team science is rising (solo 14.6%→2.5%, 21+ authors 0%→5.5% across 1995–2025); restrict longitudinal structure comparisons to post-2014 and exclude the dip window.
- Max 2,929 authors/paper (consortia) create hub nodes — filter or treat separately.

### 09 — Sentiment
- No abstract text in the published set — titles only (short, weak signal) or the local `abstracts_all` build.

### 03 / 04 — Tokenization, NER, topic modelling
- Require abstract text — local only, or scope to the permissively-licensed OA subset (`abstracts_oa_only`, a minority of records).

### Notes on the corpus itself
- **Low journal concentration** — top 100 journals are only ~19% of articles across 7,316 journals; broad coverage.
- **Abstracts lengthened ~45%** over the period (≈1,134→1,639 chars, 1994→2025) — a real trend, relevant if abstract length is ever a feature.